# Introduction Approach 1

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data**<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    pool all data per location and year for time-dependent stationary analysis<br>

---
**Additional Notes**
- Using sim_year as the actual year 

**Notes on next steps:**
- 8 models × 680 locations = 5,440 independent analyses >> parallelize code on model level

**!!! ToDo**
- plot model fits
- create a mapping function for location info and lat/lon... store as JSON to save lookup
- add Confidence Intervals for Return Levels
- add return level (and CI) to txt
- time-series for stationary analysis (keep shape and scale from global analysis)
- check why slope $\mu(t)$ != $trend$ (m/year)

# Import Libraries

In [ ]:
import time
from datetime import datetime
from glob import glob
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from numpy import isnan, unique
from pandas import DataFrame, concat

import func_gev as gev
import func_plotting as dbplt
import func_preparation as dbf
import func_utils as ut

# Settings

In [ ]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [ ]:
hindcast_start = 1960
hindcast_end = 2026

In [ ]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [ ]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

# Import data

In [ ]:
ls_files = [file for file in glob(path + '*.nc')]
ls_files

In [ ]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, trying first with `joblib` - saving ~60% (from 3min30sec down to 1min22sec)


In [ ]:
def process_model(file):
    model_name, ds_model = dbf.import_data_from_file(file) 
    ds_model_corrected = dbf.bias_correction(ds_model)
    data_valid, sites_valid, sites_total, rate_invalid = dbf.select_valid_data(
        ds_model, ds_model_corrected
    )
    ds_model.close()
    return model_name, data_valid, (sites_valid, sites_total, rate_invalid)

# ----------------------------------------------------------------------------- 
time_start1 = datetime.now()

results = Parallel(n_jobs=4)(
    delayed(process_model)(file) for file in ls_files
)

for model_name, data_valid, prep_info in results:
    dic_data_per_model[model_name]['valid data'] = data_valid
    dic_data_per_model[model_name]['preparation info'] = prep_info
    
time_end1 = datetime.now()

In [ ]:
display(Markdown("**Data Overview**"))
display(Markdown("**Model · model shape: samples (~sim_years) | ensemble members | valid locations**"))

ls_num_samples = []
dic_sim_years_per_model = dict()
for model_label in dic_data_per_model.keys():
    years = dic_data_per_model[model_label]['valid data'].sim_year.values
    unique_years = unique(years[~isnan(years)].astype(int))
    dic_sim_years_per_model[model_label] = unique_years
    data_shape = dic_data_per_model[model_label]['valid data'].shape
    
    ls_num_samples.append(data_shape[0])
    print(f"{model_label} · {data_shape}")
    
sites_valid = [dic_data_per_model[model_label]['preparation info'][0] for model_label in dic_data_per_model.keys()]
unique_years_per_model = [len(dic_sim_years_per_model[model_label]) for model_label in dic_sim_years_per_model.keys()]
sim_year_per_model_min = [min(dic_sim_years_per_model[model_label]) for model_label in dic_sim_years_per_model.keys()]
sim_year_per_model_max = [max(dic_sim_years_per_model[model_label]) for model_label in dic_sim_years_per_model.keys()]

print(
    "\nOverall, data is available from "
    f"\n\t{len(ls_files)} models, "
    f"\n\t{min(sites_valid)}-{max(sites_valid)} locations "
    f"(originally {dic_data_per_model[model_label]['preparation info'][1]})"
    f"\n\t{min(ls_num_samples)}-{max(ls_num_samples)} samples per model "
    f"\n\t - with {min(unique_years_per_model)}-{max(unique_years_per_model)} unique sim_years"
    f"\n\t - between {min(sim_year_per_model_min)}-{max(sim_year_per_model_max)}"
    )
display(Markdown(f"Execution time · {time_end1 - time_start1}sec"))

## Pool data per location cross models

from 8 models with up to 680 samples (sim_years) and two members and 3547-7216 locations, pool all data together and group per location
> Restructure multi-model ensemble by site <br>
> - from dic[model] -> shape: (sim_year, ensemble_member, location) <br>
> - to dic[location] -> shape: (sim_year, ensemble_member, model)

Also here, parallelize the code to speed up the process...

In [ ]:
da_list = []
for model_name, dic_model in dic_data_per_model.items():
    da = dic_model['valid data'] 

    da_loc = dbf.sites_to_location(da)
    da_loc = da_loc.expand_dims(model=[model_name])

    da_list.append(da_loc)

combined = xr.concat(da_list, dim="model", join="outer")

display(Markdown(f"\n**Overall, the combined dataset has the following dimensions**"))
print("Final dimensions:", combined.dims)
print("Shape:", combined.shape)
print("Number of models:", combined.model.size)
print("Number of locations:", combined.location.size)

### Validation Check

In [ ]:
list_model_labels = list(dic_data_per_model.keys())

In [ ]:
model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

display(Markdown(f"\n**Overview Original dataArray**"))
data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

display(Markdown(f"\n**Overview Revised dataArray**"))
revised_dataset, lon_rev, lat_rev = ut.get_dataset_overview_for_model_at_location(
    dic_data=combined, model_nr=model_ex, lon=lon_target, lat=lat_target
    )

In [ ]:
assert lon_target == lon_rev and lat_target == lat_rev
assert all(data_for_model_for_location.dropna() == revised_dataset.dropna())

**Learning** · make sure to never select by the site-id but always via geo-coordinates (lat | lon)

# Workflow GEV - Generalized Extreme Value

## Initial trial at 1 location, all model simulations and years

Later, batch(?) and parallelize

In [ ]:
columns_selected = ['sim_year', 'annualMax', 'lon', 'lat', 'member', 'model']

In [ ]:
loc_ex = 0

In [ ]:
data_at_location = combined[:,:,:, loc_ex].to_dataframe().dropna().reset_index()[columns_selected]
data_at_location = data_at_location.rename(columns={'annualMax':'storm_surge'})
data_at_location

---
To be continued

## Run Analysis

In [ ]:
time_start = time.time()

print("\n" + "="*70)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER MODEL")
print("="*70)

# ---------------------------------------------
df_prepared = ut.prepare_pooled_data(data=data_at_location, hindcast_start=hindcast_start, hindcast_end=hindcast_end)

lon = data_at_location.lon.unique()[0]
lat = data_at_location.lat.unique()[0]

# ---------------------------------------------
results = {}
print(f"\nAnalyzing specific location with data from {df_prepared.model.nunique()} model(s) ...")

print("\tLookup location info for batch...")
locations_label = dbf.locations_label_lookup_batched(DataFrame([lon, lat], index=['lon', 'lat']).T)
location_info = locations_label[0]
print(f"\n\tAnalyse location {location_info[0]}")

result = gev.analyze_per_location(df_prepared, lat, lon, location_info[0], return_periods)

if result is None:
    print(f"\t\t→ Warning! No results found, skipping...")
else:
    results[(lat, lon)] = result
    print(f"\t\t→ Results produced successfully; storing to dictionary...")

print("\n" + "="*70)
time_end = time.time()
print(f"✓ ANALYSIS COMPLETED IN {(time_end - time_start):.2f}s!")
print("="*70)

# Display results for one location


In [ ]:
example_location = list(results.keys())[0]
example_location

In [ ]:
result_display = results[example_location]
location_in_example = result_display['location info']

display(Markdown("\n**Subset Description**"))
print(f"number of items: {len(result_display)}")
print(f"keys: {result_display.keys()}")
print(f"values: {result_display.values()}")

#### Display Result Overview and Plots (all)

In [ ]:
ls_messages = []
print_msg = True

if result_display:
    ls_messages = ut.adding_plot_and_text("\n" + "="*100, ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(
        f"RESULTS for location (lon|lat): {result_display['location'][1]:.5f}|{result_display['location'][0]:.5f}", 
        ls_messages, print_msg
        )
    ls_messages = ut.adding_plot_and_text("="*100, ls_messages, print_msg)
    
    ls_messages = ut.adding_plot_and_text(
        f"\nClosest location identified: {location_in_example}", ls_messages, print_msg
        )
    ls_messages = ut.adding_plot_and_text(
        f"Hindcast period: {result_display['annual_maxima'].year.min().astype(int)}-"
        f"{result_display['annual_maxima'].year.max().astype(int)} "
        f"({len(result_display['annual_maxima'].year.unique())} unique years)", ls_messages, print_msg
        )
    ls_messages = ut.adding_plot_and_text(f"Observations per year: ~2 (from ensemble members)", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(
        f"Total data points for GEV: {result_display['gev_stationary']['n_obs']}", ls_messages, print_msg
        )
    ls_messages = ut.adding_plot_and_text("\nSTATIONARY GEV", ls_messages, print_msg)
    stat = result_display['gev_stationary']
    ls_messages = ut.adding_plot_and_text(f"  μ (location) = {stat['location']:.3f}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"  σ (scale) = {stat['scale']:.3f}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"  ξ (shape) = {stat['shape']:.3f}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"  Type: {stat['dist_type']}", ls_messages, print_msg)
    ls_messages = ut.adding_plot_and_text(f"  Return Level", ls_messages, print_msg)
    for x, v in result_display['return_levels_stationary'].items():
        aep = 100/int(x.split('-')[0])
        ls_messages = ut.adding_plot_and_text(
            f"  \tFor {x} period: {v:.3f} m ({aep}% annual exceedance probability)", ls_messages, print_msg
            )
        
    if result_display['gev_nonstationary']:
        ls_messages = ut.adding_plot_and_text("\nNON-STATIONARY GEV", ls_messages, print_msg)
        nonstat = result_display['gev_nonstationary']
        ls_messages = ut.adding_plot_and_text(f"  μ(t) = {nonstat['mu0']:.3f} + {nonstat['mu1']:.4f}·t", ls_messages, print_msg)
        ls_messages = ut.adding_plot_and_text(f"  Trend = {nonstat['mu1'] * nonstat['years_std']:.4f} m/year", ls_messages, print_msg)
        ls_messages = ut.adding_plot_and_text(f"  Return Level Evolution", ls_messages, print_msg)
        for x,y in zip(
            result_display['return_levels_nonstationary_start']['values'].items(), 
            result_display['return_levels_nonstationary_end']['values'].items()
            ):

            period = x[0]
            aep = 100/int(period.split('-')[0])
            ls_messages = ut.adding_plot_and_text(
                f"  \tFor {period} period: {x[1]:.3f}m - {y[1]:.3f}m ({aep}% annual exceedance probability)",
                ls_messages, print_msg
            )
            
    if result_display['model_comparison']:
        ls_messages = ut.adding_plot_and_text("\nMODEL COMPARISON", ls_messages, print_msg)
        comp = result_display['model_comparison']
        ls_messages = ut.adding_plot_and_text(f"  p-value: {comp['p_value']:.4f}", ls_messages, print_msg)
        ls_messages = ut.adding_plot_and_text(f"  Decision: {comp['decision']}", ls_messages, print_msg)
        ls_messages = ut.adding_plot_and_text(f"  → {comp['recommendation']}", ls_messages, print_msg)

    # -----------------------------------------------------------------------------------------------------------
    today_ = str(datetime.today().date().isoformat())
    save_path = path_export + today_
    Path(save_path).mkdir(parents=True, exist_ok=True)            
    country = result_display['location info'].split(',')[-1].strip()
    lat_str = str(round(float(result_display['location'][0]), 3))
    lon_str = str(round(float(result_display['location'][1]),3))
    
    with open(save_path + f"/GEVanalysis_{country}_{lat_str}|{lon_str}_{today_}.txt", 'w') as f:
        f.write('\n'.join(ls_messages))
    
    print("\nVISUALIZE RESULTS")
    dbplt.plot_pooled_analysis(
        results=results, 
        lat_lon_tuple=result_display['location'], 
        location_info=result_display['location info'],
        periods_evolution = plot_period_evolution,
        box_parameters_x=0.05, box_parameters_y=0.95,
        width_bar_returns=0.35,
        leg_comparison_x=0.35, leg_comparison_y=0.65, linespace=1.5,
        save_path = save_path,
        color_markers='#99E3DDFF', 
        colors_trends='#1D141BFF', 
        colors_models=['#B887ADFF', '#008A80FF'],
        colors_return_levels=['#008A80FF','#CAA5C2FF'],
        bbox_color='#F5F5F5FF',
        axes_color='#333333', 
        linestyle_trends=['dashdot', 'dashed', 'solid'], 
        fontsize=12, figsize=(15, 7.5),
    )